# Notebook 01: EGFP mRNA Design Exploration

This notebook demonstrates the **Agentic mRNA Design Pipeline** on the EGFP protein.
It covers:

1. Seeding candidates with different strategies (CAI-max, GC-balanced, harmonization)
2. Computing and visualising the initial objective scores
3. Running 20 iterations of the Rule-Based controller
4. Tracking hypervolume improvement over iterations
5. Comparing Pareto-optimal sequences

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Imports OK')

## 1. Load EGFP Target

In [ ]:
with open('../data/targets/benchmark_proteins.json') as f:
    targets = json.load(f)['targets']

egfp = next(t for t in targets if t['name'] == 'EGFP')
protein = egfp['protein']
utr5 = egfp.get('utr5', '')
utr3 = egfp.get('utr3', '')

print(f'Protein: {protein[:30]}... ({len(protein)} aa)')
print(f"5'UTR length: {len(utr5)} nt")
print(f"3'UTR length: {len(utr3)} nt")

## 2. Seed Population with Multiple Strategies

In [ ]:
from mrna_design.designer.seeds import (
    seed_cai_max, seed_gc_balanced, seed_harmonized
)
from mrna_design.metrics.aggregator import compute_all

strategies = {
    'CAI-Max':      seed_cai_max(protein, utr5=utr5, utr3=utr3),
    'GC-Balanced':  seed_gc_balanced(protein, utr5=utr5, utr3=utr3),
    'Harmonized':   seed_harmonized(protein, utr5=utr5, utr3=utr3),
}

seeds = {}
for name, cand in strategies.items():
    scores, diags = compute_all(cand, run_safety=False)
    cand = cand.model_copy(update={'scores': scores, 'diagnostics': diags})
    seeds[name] = cand
    print(f'{name:15s}  CAI={scores.cai:.3f}  GC={scores.gc_content:.3f}  MFE={scores.mfe:.1f}  CpG={scores.cpg_density:.3f}')

## 3. Visualise Initial Score Profiles

In [ ]:
metrics = ['cai', 'gc_content', 'cpg_density', 'upa_density', 'uridine_fraction']
labels  = ['CAI', 'GC Content', 'CpG Density', 'UpA Density', 'Uridine Fraction']

fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4))
colors = sns.color_palette('husl', len(seeds))

for ax, metric, label in zip(axes, metrics, labels):
    vals  = [getattr(cand.scores, metric, 0) or 0 for cand in seeds.values()]
    bars  = ax.bar(list(seeds.keys()), vals, color=colors, edgecolor='white', linewidth=1.2)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xticklabels(list(seeds.keys()), rotation=25, ha='right', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('EGFP Seed Comparison — Initial Objective Profiles', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('seed_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved seed_profiles.png')

## 4. Run 20-Iteration Optimisation with Rule-Based Controller

In [ ]:
from mrna_design.optimize import OptimizeConfig, optimize
from mrna_design.controller.rule_based import RuleBasedController
from pathlib import Path

cfg = OptimizeConfig(
    max_iters=20,
    population_k=5,
    rng_seed=42,
    run_safety=False,
    include_lineardesign=False,
    output_dir=Path('results/egfp_explore'),
)

ctrl = RuleBasedController(rng_seed=42)
print('Running 20-iteration Rule-Based optimisation on EGFP...')
archive = optimize(protein, ctrl, utr5=utr5, utr3=utr3, config=cfg)

print(f'\nDone! Archive size: {len(archive)} Pareto-optimal candidates')
print(f'Final hypervolume: {archive.hypervolume():.6f}')

## 5. Plot Hypervolume Trajectory

In [ ]:
hv_history = archive.hv_history()
iters = [h[0] for h in hv_history]
hvs   = [h[1] for h in hv_history]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(iters, hvs, color='#4E79A7', linewidth=2.5, marker='o', markersize=5, label='Rule-Based Controller')
ax.fill_between(iters, 0, hvs, alpha=0.15, color='#4E79A7')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Hypervolume', fontsize=12)
ax.set_title('EGFP Hypervolume Trajectory (Rule-Based, 20 iterations)', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('hv_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved hv_trajectory.png')

## 6. Pareto Front: CAI vs CpG Density

In [ ]:
pareto_cais = [e.candidate.scores.cai or 0 for e in archive._entries]
pareto_cpgs = [e.candidate.scores.cpg_density or 0 for e in archive._entries]
pareto_iters = [e.iteration for e in archive._entries]

# Also plot initial seeds for comparison
seed_cais = [cand.scores.cai or 0 for cand in seeds.values()]
seed_cpgs = [cand.scores.cpg_density or 0 for cand in seeds.values()]

fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(pareto_cais, pareto_cpgs, c=pareto_iters, cmap='viridis',
                s=80, zorder=3, edgecolors='white', linewidths=0.8, label='Pareto Archive')
plt.colorbar(sc, ax=ax, label='Iteration discovered')

ax.scatter(seed_cais, seed_cpgs, marker='*', s=250, c='tomato',
           zorder=4, edgecolors='white', linewidths=0.8, label='Seeds')

for name, cai, cpg in zip(seeds.keys(), seed_cais, seed_cpgs):
    ax.annotate(name, (cai, cpg), textcoords='offset points',
                xytext=(8, 4), fontsize=8, color='tomato')

ax.set_xlabel('CAI (higher = better)', fontsize=12)
ax.set_ylabel('CpG Density (lower = better)', fontsize=12)
ax.set_title('EGFP Pareto Front: CAI vs CpG Density', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('pareto_front.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pareto_front.png')

## 7. Top-5 Pareto Candidates

In [ ]:
import pandas as pd

rows = []
for e in archive._entries[:10]:
    s = e.candidate.scores
    rows.append({
        'ID':           e.candidate.sequence_id[:10],
        'Iter':         e.iteration,
        'CAI':          round(s.cai or 0, 4),
        'MFE':          round(s.mfe or 0, 2),
        'GC':           round(s.gc_content or 0, 4),
        'CpG/100nt':    round(s.cpg_density or 0, 4),
        'UpA/100nt':    round(s.upa_density or 0, 4),
        'miRNA hits':   s.mirna_seed_hits or 0,
        'Accepted edits': len(e.candidate.accepted_edits),
    })

df = pd.DataFrame(rows)
print('Top-10 Pareto Archive Candidates:')
print(df.to_string(index=False))

## 8. Sliding-Window CAI for Best Candidate

In [ ]:
from mrna_design.metrics.cai import sliding_window_cai

best = archive._entries[0].candidate
cds  = best.cds
win_cais = sliding_window_cai(cds, window=30)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(win_cais, color='#59A14F', linewidth=1.8, label='Sliding-window CAI (30-codon)')
ax.axhline(np.mean(win_cais), color='#E15759', linestyle='--', linewidth=1.2,
           label=f'Mean CAI = {np.mean(win_cais):.3f}')
ax.fill_between(range(len(win_cais)), win_cais, alpha=0.2, color='#59A14F')
ax.set_xlabel('Codon window start', fontsize=12)
ax.set_ylabel('Window CAI', fontsize=12)
ax.set_title(f'Sliding-Window CAI — Best EGFP Candidate ({best.sequence_id[:10]})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('sliding_cai.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved sliding_cai.png')